# Phase 1 — baseline training run (fp16, timm init)

Two fixes over the published script, both found in Phase 0:

1. **fp16 instead of bf16.** T4 is sm_75 with no native bf16; measured
   `bf16 796.6 ms/step, fp16 254.4 ms/step, fp32 607.8 ms/step` at 384/bs4 — bf16 is
   3.13x slower than fp16 and slower even than fp32. Autocast now picks bf16 only where
   the GPU supports it natively, and adds a `GradScaler` when running fp16.
2. **`--ckpt timm`.** The script defaults to an SSL-pretrained backbone
   `ckpt/raptor_ssl_last.pt` that is not published, so the run died immediately.
   `timm` is the script's own supported fallback (ImageNet-pretrained CoAtNet).

This means the model cannot exactly reproduce theirs — they started from SSL weights we
do not have. Timing is measured first on a capped subset, then a full run is sized to fit
inside one 12 h session, since the script has no `--resume`.

In [ ]:
TRAIN_PY_B64 = (
    '...'
)

import base64, pathlib
pathlib.Path('/kaggle/working/train_knee.py').write_bytes(base64.b64decode(TRAIN_PY_B64))
print('wrote train_knee.py')

In [ ]:
import glob, subprocess, sys, time, json, os
def _find(name, root='/kaggle/input'):
    for r, d, f in os.walk(root):
        d[:] = [x for x in d if x not in ('train_series', 'test_series')]
        if name in f:
            return os.path.join(r, name)
    raise FileNotFoundError(name)
LAB_PARQUET = _find('labels_llm_soft.parquet')
print('labels:', LAB_PARQUET)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
BASE = [sys.executable, '/kaggle/working/train_knee.py',
        '--arch', 'coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k', '--res', '384',
        '--grad_ckpt', '--ckpt', 'timm', '--labels', LAB_PARQUET]

# T4 has 14.56 GiB; the reference recipe (bs 8, k 12) was tuned for a 24 GB 4090 and
# OOMs here. Memory scales with bs*k windows through the backbone, so step down until
# one fits, then keep that shape for the real run.
LADDER = [(4, 12, 16), (4, 8, 16), (2, 12, 12), (2, 8, 12)]
FIT = None

def run(extra, tag, timeout=None):
    cmd = BASE + extra + ['--tag', tag]
    print('\n$ ' + ' '.join(cmd[1:]), flush=True)
    t0 = time.time()
    p = subprocess.run(cmd, cwd='/kaggle/working', capture_output=True, text=True, timeout=timeout)
    el = time.time() - t0
    print(f'--- exit={p.returncode} in {el/60:.1f} min ---')
    print(p.stdout[-4000:])
    if p.returncode != 0: print('STDERR:', p.stderr[-3000:])
    return p.returncode, el

In [ ]:
# Timing probe: find a batch shape that fits, then measure 1 epoch on 300 studies.
import re
for bs, k, ke in LADDER:
    rc, el = run(['--epochs','1','--limit','300','--bs',str(bs),'--k',str(k),
                  '--k_eval',str(ke)], f'timing_b{bs}k{k}')
    if rc == 0:
        FIT = (bs, k, ke); break
    print(f'  bs={bs} k={k} did not fit; stepping down', flush=True)
assert FIT is not None, 'no batch shape fit on this GPU'
bs, k, ke = FIT
per = el / 300
full_ep = per * 4349
print(f'\n  FIT bs={bs} k={k} k_eval={ke}')
print(f'  {per:.2f} s/study | projected full epoch {full_ep/60:.1f} min')
budget = 10.0 * 3600
EPOCHS = max(1, int(budget // full_ep))
print(f'  epochs that fit in 10 h: {EPOCHS}')
json.dump({'fit': FIT, 'per_study_s': per, 'full_epoch_s': full_ep,
           'epochs_planned': EPOCHS}, open('/kaggle/working/phase1_timing.json','w'), indent=2)


In [ ]:
# The real run, sized to the measured budget. Saves best-by-gold-AUC + top-3 SWA.
rc, el = run(['--epochs', str(EPOCHS), '--bs', str(bs), '--k', str(k), '--k_eval', str(ke)], 'p1')
print(f'\ntraining finished rc={rc} in {el/3600:.2f} h')
import glob, json, os
for f in sorted(glob.glob('/kaggle/working/*.pt') + glob.glob('/kaggle/working/*.json') +
                glob.glob('/kaggle/working/*.npz')):
    print(f'  {os.path.basename(f):34s} {os.path.getsize(f)/1e6:8.1f} MB')
for f in glob.glob('/kaggle/working/raptor_*_p1*.json'):
    print(json.dumps(json.load(open(f)), indent=2)[:1200])

In [ ]:
# Report the gold-gate result against the published checkpoints.
import torch, glob, os
PUBLISHED = {'raptor_ft_coatnet_v5_full_swa.pt':0.9214,'raptor_ft_coatnet_v10_full.pt':0.9174,
             'raptor_ft_coatnet_v4_full.pt':0.9167,'raptor_ft_coatnet384x.pt':0.9054}
mine = sorted(glob.glob('/kaggle/working/raptor_ft_p1*.pt'))
print('published gold-gate AUCs (58 held-out studies):')
for k,v in sorted(PUBLISHED.items(), key=lambda x:-x[1]): print(f'  {v:.4f}  {k}')
print()
for m in mine:
    ck = torch.load(m, map_location='cpu', weights_only=False)
    g = ck.get('gold_auc')
    print(f'  {g:.4f}  {os.path.basename(m)}  (ours, epoch {ck.get("epoch")}, src={ck.get("src")})')
    if isinstance(ck.get('aucs'), dict):
        for t,v in ck['aucs'].items(): print(f'        {t:18s} {v:.4f}')
    del ck